#### Simple Gen AI APP Using Langchain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [3]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
loader=WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administrators/manage_spend")
loader

In [5]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIToolsPolly AI assistantBetaCLISkillsAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.\nIt helps you trace requests, evaluate outputs, test prompts, and manage deployments in one place. LangSmith works with any agent stack, whethe

In [6]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [7]:
documents

[Document(metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIToolsPolly AI assistantBetaCLISkillsAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.\nIt helps you trace requests, evaluate outputs, test prompts, and manage deployments in one place. LangSmith works with any agent stack, whethe

In [8]:
from langchain_community.embeddings import OllamaEmbeddings

embeddings=(
    OllamaEmbeddings(model="gemma2:2b")  ##by default it ues llama2
)


C:\Users\KRISHNA SRIVASTAVA\AppData\Local\Temp\ipykernel_2640\3743117711.py:4: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  OllamaEmbeddings(model="gemma2:2b")  ##by default it ues llama2


In [9]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [10]:
vectorstoredb

In [11]:
## Query From a vector db
query="LangSmith has two usage limits: total traces and extended"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'\u200bGet started\nCreate an accountSign up at smith.langchain.com (no credit card required).\nYou can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → Create API Key.\nCopy the key and save it securely.Choose your integrationLangSmith works with many frameworks and providers including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.\nBrowse available integrations to connect your stack.\nOnce your account and API key are ready, choose a quickstart to begin building with LangSmith:\nObservabilityGain visibility into every step your application takes to debug faster and improve reliability.Start tracingEvaluationMeasure and track quality over time to ensure your AI applications are consistent and trustworthy.Evaluate your appDeploymentDeploy your agents as Agent Servers, ready to scale in production.Deploy your agents\n\u200bMore ways to build'

In [12]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma2:2b"
)

In [13]:
## Retrieval Chain, Document chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
| ChatOllama(model='gemma2:2b')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [14]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"LangSmith has two usage limits: total traces and extended",
    "context":[Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph. ")]
})

'What are the two usage limits for LangSmith? \n'

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [15]:
### Input--->Retriever--->vectorstoredb

vectorstoredb

In [16]:
retriever=vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [17]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001CD7F9F6C20>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
            | 

In [18]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"LangSmith has two usage limits: total traces and extended"})
response['answer']

"Based on the provided context, LangSmith is a framework-agnostic platform designed for developing, debugging, and deploying AI agents and LLM applications. \n\nHere's a breakdown:\n\n* **Framework Agnostic:**  LangSmith works with any agent stack, whether you're using an existing framework like OpenAI or building your own.\n* **Integrated Workflow:**  It combines observability (e.g., tracing requests), evaluation (e.g., measuring outputs), deployment, and platform setup in one seamless workflow.\n* **Local Development & Production:** You can prototype locally and then move to production with integrated monitoring and evaluation.\n* **Tools for Prompt Engineering:** LangSmith includes tools for testing and iterating on prompts, allowing for faster development and deployment.\n* **Security & Compliance:** LangSmith meets HIPAA, SOC 2 Type 2, and GDPR compliance standards.\n\n**To summarize:** LangSmith aims to simplify the entire process of building and deploying AI agents, providing a 

In [19]:

response

{'input': 'LangSmith has two usage limits: total traces and extended',
 'context': [Document(id='1ad6f74a-2874-4708-8343-417348f17915', metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='\u200bGet started\nCreate an accountSign up at smith.langchain.com (no credit card required).\nYou can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → Create API Key.\nCopy the key and save it securely.Choose your integrationLangSmith works with many frameworks and providers including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.\nBrowse available integrations to connect your stack.\nOnce your account and API key are ready, choose a quickstart to begin building with LangSmith:\nObservabilityGain visibility into every step your application takes to debug faster and improve reliability.Start tracingEvaluationMeasure and t

In [20]:
response['context']

[Document(id='1ad6f74a-2874-4708-8343-417348f17915', metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='\u200bGet started\nCreate an accountSign up at smith.langchain.com (no credit card required).\nYou can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → Create API Key.\nCopy the key and save it securely.Choose your integrationLangSmith works with many frameworks and providers including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.\nBrowse available integrations to connect your stack.\nOnce your account and API key are ready, choose a quickstart to begin building with LangSmith:\nObservabilityGain visibility into every step your application takes to debug faster and improve reliability.Start tracingEvaluationMeasure and track quality over time to ensure your AI applications are consistent and trustworth